<a href="https://colab.research.google.com/github/thinethwic/ai-in-eCommerce-salary-predicted-model/blob/developer/E_Commerce_Customers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
df.head()

Saving Dataset salary 2024.csv to Dataset salary 2024.csv


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2024,SE,FT,AI Engineer,202730,USD,202730,US,0,US,M
1,2024,SE,FT,AI Engineer,92118,USD,92118,US,0,US,M
2,2024,SE,FT,Data Engineer,130500,USD,130500,US,0,US,M
3,2024,SE,FT,Data Engineer,96000,USD,96000,US,0,US,M
4,2024,SE,FT,Machine Learning Engineer,190000,USD,190000,US,0,US,M


In [27]:
df.tail()

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
16529,2020,SE,FT,Data Scientist,412000,USD,412000,US,100,US,L
16530,2021,MI,FT,Principal Data Scientist,151000,USD,151000,US,100,US,L
16531,2020,EN,FT,Data Scientist,105000,USD,105000,US,100,US,S
16532,2020,EN,CT,Business Data Analyst,100000,USD,100000,US,100,US,L
16533,2021,SE,FT,Data Science Manager,7000000,INR,94665,IN,50,IN,L


In [2]:
print("\n ----Data Information Summary----")
df.info()


 ----Data Information Summary----
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16534 entries, 0 to 16533
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   work_year           16534 non-null  int64 
 1   experience_level    16534 non-null  object
 2   employment_type     16534 non-null  object
 3   job_title           16534 non-null  object
 4   salary              16534 non-null  int64 
 5   salary_currency     16534 non-null  object
 6   salary_in_usd       16534 non-null  int64 
 7   employee_residence  16534 non-null  object
 8   remote_ratio        16534 non-null  int64 
 9   company_location    16534 non-null  object
 10  company_size        16534 non-null  object
dtypes: int64(4), object(7)
memory usage: 1.4+ MB


In [3]:
print(df.isnull().sum())

work_year             0
experience_level      0
employment_type       0
job_title             0
salary                0
salary_currency       0
salary_in_usd         0
employee_residence    0
remote_ratio          0
company_location      0
company_size          0
dtype: int64


In [4]:
X = df[["experience_level","employment_type","job_title","salary","salary_currency"]]
y = df["salary_in_usd"]

In [5]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16534 entries, 0 to 16533
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   experience_level  16534 non-null  object
 1   employment_type   16534 non-null  object
 2   job_title         16534 non-null  object
 3   salary            16534 non-null  int64 
 4   salary_currency   16534 non-null  object
dtypes: int64(1), object(4)
memory usage: 646.0+ KB


In [6]:
y.info()

<class 'pandas.core.series.Series'>
RangeIndex: 16534 entries, 0 to 16533
Series name: salary_in_usd
Non-Null Count  Dtype
--------------  -----
16534 non-null  int64
dtypes: int64(1)
memory usage: 129.3 KB


In [7]:
numerical_features = ['salary']
categorical_features = ['experience_level','employment_type','salary_currency']

print("\n Numerical to be Scaled \n")
print(numerical_features)
print("\n Categorical to be Encoded \n")
print(categorical_features)


 Numerical to be Scaled 

['salary']

 Categorical to be Encoded 

['experience_level', 'employment_type', 'salary_currency']


In [8]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X, y,  test_size=0.2, random_state=42)
print(f"Training data Shape {X_train.shape}")
print(f"Testing data Shape {X_test.shape}")

Training data Shape (13227, 5)
Testing data Shape (3307, 5)


In [9]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('one', OneHotEncoder(), categorical_features),
        ('scaler', StandardScaler(), numerical_features)
    ]
)

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

lr_pipline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])
lr_pipline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('one', OneHotEncoder(),
                                                  ['experience_level',
                                                   'employment_type',
                                                   'salary_currency']),
                                                 ('scaler', StandardScaler(),
                                                  ['salary'])])),
                ('regressor', LinearRegression())])

In [11]:
lr_preditc = lr_pipline.predict(X_test)

In [12]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

r2_lr = r2_score(y_test, lr_preditc)
mse_lr = mean_squared_error(y_test, lr_preditc)
rmse_lr = np.sqrt(mse_lr)

print(f"R-squared (R²): {r2_lr:.4f}")
print(f"Mean Squared Error (MSE): {mse_lr:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_lr:.4f}")
print(f"\nInterpretation: The model explains {r2_lr:.1%} of the variance in the log-transformed price.")

R-squared (R²): 0.4166
Mean Squared Error (MSE): 2789716092.9035
Root Mean Squared Error (RMSE): 52817.7630

Interpretation: The model explains 41.7% of the variance in the log-transformed price.


In [13]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

dt_pipline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', DecisionTreeRegressor(random_state=42))
])
dt_pipline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('one', OneHotEncoder(),
                                                  ['experience_level',
                                                   'employment_type',
                                                   'salary_currency']),
                                                 ('scaler', StandardScaler(),
                                                  ['salary'])])),
                ('regressor', DecisionTreeRegressor(random_state=42))])

In [14]:
dt_predict = dt_pipline.predict(X_test)

In [15]:
r2_dt = r2_score(y_test, dt_predict)
mse_dt = mean_squared_error(y_test, dt_predict)
rmse_dt = np.sqrt(mse_lr)

print(f"R-squared (R²): {r2_dt:.4f}")
print(f"Mean Squared Error (MSE): {mse_dt:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_dt:.4f}")
print(f"\nInterpretation: The model explains {r2_dt:.1%} of the variance in the log-transformed price.")

R-squared (R²): 0.9888
Mean Squared Error (MSE): 53609340.5328
Root Mean Squared Error (RMSE): 52817.7630

Interpretation: The model explains 98.9% of the variance in the log-transformed price.


In [17]:
rf_pipline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1))]
)
rf_pipline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('one', OneHotEncoder(),
                                                  ['experience_level',
                                                   'employment_type',
                                                   'salary_currency']),
                                                 ('scaler', StandardScaler(),
                                                  ['salary'])])),
                ('regressor',
                 RandomForestRegressor(n_estimators=50, n_jobs=-1,
                                       random_state=42))])

In [18]:
re_predict = rf_pipline.predict(X_test)

In [19]:
r2_rf = r2_score(y_test, re_predict)
mse_rf = mean_squared_error(y_test, re_predict)
rmse_rf = np.sqrt(mse_lr)

print(f"R-squared (R²): {r2_rf:.4f}")
print(f"Mean Squared Error (MSE): {mse_rf:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_rf:.4f}")
print(f"\nInterpretation: The model explains {r2_rf:.1%} of the variance in the log-transformed price.")

R-squared (R²): 0.9903
Mean Squared Error (MSE): 46236305.0106
Root Mean Squared Error (RMSE): 52817.7630

Interpretation: The model explains 99.0% of the variance in the log-transformed price.


In [20]:
model_performance = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest'],
    'R-squared (R²)': [r2_lr, r2_dt, r2_rf],
    'RMSE': [rmse_lr, rmse_dt, rmse_rf]
}).sort_values(by='R-squared (R²)', ascending=False)

print("--- Model Performance Comparison ---")
print(model_performance)

print("\n🏆 BEST MODEL SELECTION 🏆")
print("The Random Forest Regressor is the best model. It has the highest R-squared and the lowest RMSE,")
print("indicating it makes the most accurate predictions on unseen data.")

--- Model Performance Comparison ---
               Model  R-squared (R²)          RMSE
2      Random Forest        0.990330  52817.763043
1      Decision Tree        0.988789  52817.763043
0  Linear Regression        0.416581  52817.763043

🏆 BEST MODEL SELECTION 🏆
The Random Forest Regressor is the best model. It has the highest R-squared and the lowest RMSE,
indicating it makes the most accurate predictions on unseen data.


In [22]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [23]:
import joblib
filename = '/content/drive/MyDrive/e-commerce.joblib'
joblib.dump(rf_pipline, filename)

['/content/drive/MyDrive/e-commerce.joblib']

In [24]:
import joblib
filename = 'e-commerce.joblib'
joblib.dump(rf_pipline, filename)

['e-commerce.joblib']

In [26]:
import pandas as pd

# Create a sample DataFrame for real-world input
# Ensure the columns match the features used for training (X)
real_world_data = pd.DataFrame({
    'experience_level': ['SE', 'MI', 'EN'],
    'employment_type': ['FT', 'CT', 'PT'],
    'job_title': ['Data Scientist', 'Software Engineer', 'Data Analyst'],
    'salary': [120000, 80000, 50000],
    'salary_currency': ['USD', 'EUR', 'GBP']
})

print("--- New Real-World Input Data ---")
display(real_world_data)

# Make predictions using the trained Random Forest pipeline
real_world_predictions = rf_pipline.predict(real_world_data)

print("\n--- Predicted Salaries (in USD) ---")
for i, pred in enumerate(real_world_predictions):
    print(f"Predicted salary for input {i+1}: {pred:.2f} USD")


--- New Real-World Input Data ---


,experience_level,employment_type,job_title,salary,salary_currency
0,SE,FT,Data Scientist,120000,USD
1,MI,CT,Software Engineer,80000,EUR
2,EN,PT,Data Analyst,50000,GBP



--- Predicted Salaries (in USD) ---
Predicted salary for input 1: 120000.00 USD
Predicted salary for input 2: 84379.04 USD
Predicted salary for input 3: 62065.46 USD


In [25]:
import joblib
import pandas as pd

# Path to the saved model in Google Drive
model_path = '/content/drive/MyDrive/e-commerce.joblib'

# Load the trained model
loaded_rf_pipeline = joblib.load(model_path)

print("Model loaded successfully from Google Drive.")

# Create a sample DataFrame for real-world input (re-creating for clarity)
# Ensure the columns match the features used for training (X)
real_world_data = pd.DataFrame({
    'experience_level': ['SE', 'MI', 'EN'],
    'employment_type': ['FT', 'CT', 'PT'],
    'job_title': ['Data Scientist', 'Software Engineer', 'Data Analyst'],
    'salary': [120000, 80000, 50000],
    'salary_currency': ['USD', 'EUR', 'GBP']
})

print("\n--- New Real-World Input Data ---")
display(real_world_data)

# Make predictions using the loaded Random Forest pipeline
loaded_model_predictions = loaded_rf_pipeline.predict(real_world_data)

print("\n--- Predicted Salaries (in USD) using loaded model ---")
for i, pred in enumerate(loaded_model_predictions):
    print(f"Predicted salary for input {i+1}: {pred:.2f} USD")

Model loaded successfully from Google Drive.

--- New Real-World Input Data ---


,experience_level,employment_type,job_title,salary,salary_currency
0,SE,FT,Data Scientist,120000,USD
1,MI,CT,Software Engineer,80000,EUR
2,EN,PT,Data Analyst,50000,GBP



--- Predicted Salaries (in USD) using loaded model ---
Predicted salary for input 1: 120000.00 USD
Predicted salary for input 2: 84379.04 USD
Predicted salary for input 3: 62065.46 USD
